# Codebase walkthrough

Run from the repository root.


See [DEFINITIONS.md](DEFINITIONS.md) for the time grid, notation, and array shapes referenced in the code comments.

In [2]:
# sys.path controls where Python looks for imports.
import sys
from pathlib import Path
import numpy as np

# Add the repo’s src directory so this notebook can import the local package.
sys.path.insert(0, str(Path("src").resolve()))
from gauge_arbitrage.vf import vf_series
from gauge_arbitrage.vf import detector

## Simulate a shared shock

The risky assets have exposure to a shock. Their spread cancels that shock, and the model plants drift along the null direction. X_mu_43 scales the standard-normal draws and returns prices with time on rows.

In [4]:
model = vf_series.Vf_series(200, 400, seed=42)
# T = Total amount of time
# n = Total number of steps
# seed = random seed

sigma = np.array([[0.0], [0.2], [0.2]])
# volatility matrix for 3 assets, shape (N, d) where
# N = number of assets
# d = number of independent Brownian motions


Z = model.rng.standard_normal((model.T, 1))
# Draw one standard-normal value per step and shock with shape (T, d)
# The simulator scales these draws by sqrt(dt) to make Brownian increments

X = model.X_mu_43(sigma=sigma, beta=np.array([0.25]), alpha_A=np.array([0.04]), X_0=np.array([1., 100., 100.]), Z=Z)
# beta = price of risk for each shock
# alpha_A = planted null-portfolio drift
# X_0 = initial prices in asset order 
# Z = the Brownian increments
# X is the price matrix with shape (n+1, N), with the initial prices in row 0 and cash in column 0

J = vf_series.calc_J(sigma)[:, 0]
# J selects the first theoretical null direction from the (N, k) column basis
# Its weights sum to zero and its exposure to every shock is zero

L, dt = 40, model.dt
# L = number of past returns per covariance fit
# dt = T/n

print("Price shape:", X.shape)
print("Theoretical direction:", J)
# The theoretical target is the sum of squared planted null drifts. Here k=1.
print("Planted squared drift rate:", 0.04**2)

Price shape: (401, 3)
Theoretical direction: [-1.00875766e-16 -7.07106781e-01  7.07106781e-01]
Planted squared drift rate: 0.0016


## Run the detector

The fit uses past log returns. Each aligned basis scores the next simple return divided by the time step. The first consecutive product ends at price index L+2. Estimated basis vectors are stored as rows.

In [9]:
#Simplified detector (run_estimator()). For the one that uses returns, use run_detector()
detector_1 = detector.run_estimator(X, L=L, k=1, dt=dt)
# Fit k=1 calm direction using the previous L log returns at each step

A2, alpha_hat, lambdas, J_hat = detector_1["A2"], detector_1["alpha_hat"], detector_1["lambdas"], detector_1["J_hat"]
# A2 = the detector value per price index
# alpha_hat = the estimated drift per price index
# lambdas = the estimated eigenvalues per price index
# J_hat = the estimated null direction per price index

first = np.flatnonzero(np.isfinite(A2))[0]
print("First detector index:", first)
# Find the first price index with a finite detector product.

print("First fitted basis:", J_hat[0])
# J_hat starts with the fit at L

print("Mean rolling score:", np.nanmean(A2))
# Average the available products, skipping the initial NaNs

First detector index: 42
First fitted basis: [[ 3.64035188e-16 -7.07106781e-01  7.07106781e-01]]
Mean rolling score: 0.001678069280220519


## Compute the core identity

In [7]:
R = detector.calc_R(X)[L:]
# row i of R measures the log return from X[i] to X[i+1]

#OLD CODE. New code is in predict.py. You need to make functions to calculate Gamma0 and Gamma1 for each simulation.

#D1, se = vf.D_direct(R, J)
# D1 = E[h_t * h_t+1] the core identity
# se = standard error of the mean

#D2, drift, lag = vf.D_formula(R, J)
# total = (J^T(Gamma_1+mumu^T)J) the transformed core identity
# drift = (J^T mu)^2 the dift part
# lag = J^T Gamma_1 J the lag covariance part

#print(f"D1={D1:.6g}, D2={D2:.6g}")
#print(f"Drift part={drift:.6g}, Lag Part={lag:.6g}")

## Compare pricing units

Divide each row by the price of the chosen asset and rerun the same estimator to check for invariance.

In [8]:
for asset, name in enumerate(["Cash", "Asset 1", "Asset 2"]):
    relative_prices = X / X[:, [asset]]
    # The chosen asset’s relative price is always one
    
    detector_2 = detector.run_estimator(relative_prices, L=L, k=1, dt=dt)["A2"]
    # Refit and score on the rescaled prices with the same L, k, and dt

    print(f"{name}: {np.nanmean(detector_2):.6g}")
    # Compare the mean detector values across pricing units

Cash: 0.00167807
Asset 1: 0.00164601
Asset 2: 0.00155548
